# 04 - Gene annotations and pathway-coloured embeddings

Item 4 of the brief: what pathway does each gene belong to, and is it a ligand, a receptor or a
transcription factor?

**Where the evidence comes from.** Every flag is a query against a public resource, not a
hand-typed list - the exact query strings are kept in `GeneAnnotation.sources` so a reviewer can
re-run them:

| field | evidence |
|---|---|
| `pathways`, `top_pathway` | KEGG mouse pathway membership (`rest.kegg.jp/link/mmu/pathway`) |
| `is_tf` | GO:0003700 DNA-binding transcription factor activity / UniProt KW-0804-5 |
| `is_receptor` | GO:0038023 signaling receptor activity / UniProt KW-0675 |
| `is_ligand` | GO:0048018 receptor ligand activity, GO:0005102 signaling receptor binding |

`top_pathway` is the *smallest* pathway a gene belongs to, not the first one returned - otherwise
every gene lands in "Metabolic pathways" and colouring by pathway tells you nothing.

**Coverage is a result.** Annotation is incomplete for any gene list; the counts printed below are
part of the output, not a footnote.

In [ ]:
import os, sys, warnings
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
os.chdir(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else None
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np, pandas as pd
from panelprep import Config
print('panelprep ready; cwd =', os.getcwd())

In [ ]:
from panelprep.annotate import annotate_genes

GENES = ['Sox2','Pou5f1','Nanog','T','Foxa2','Gata4','Gata6','Hand1','Hand2','Mesp1','Mixl1',
         'Eomes','Lefty1','Lefty2','Nodal','Bmp4','Wnt3','Wnt3a','Fgf8','Fgfr1','Cer1','Sox17',
         'Krt8','Krt18','Mest','Dnmt3b','Pdgfra','Cdx2','Tbx6','Msgn1','Otx2','Hesx1','Notch1',
         'Dll1','Acvr1','Bmpr1a','Tgfbr1','Smad1','Smad4','Ctnnb1']

ann = annotate_genes(GENES, sources=['kegg','roles'], cache_dir='data/external', offline=False)
ann.table.head(15)

In [ ]:
print('coverage over', len(GENES), 'genes:')
print('  with any pathway      :', int((ann.table['pathways'].astype(str).str.len() > 0).sum()))
print('  is_tf                 :', int(ann.table['is_tf'].sum()))
print('  is_receptor           :', int(ann.table['is_receptor'].sum()))
print('  is_ligand             :', int(ann.table['is_ligand'].sum()))
print()
print('top_pathway distribution:')
ann.table['top_pathway'].value_counts().head(15)

### Spot-check the role calls

These should be right by inspection: `T`/`Sox2`/`Gata4`/`Eomes` are transcription factors, `Fgfr1`/`Notch1`/`Bmpr1a` are receptors, `Wnt3`/`Bmp4`/`Nodal`/`Dll1` are ligands.

In [ ]:
check = ['T','Sox2','Gata4','Eomes','Fgfr1','Notch1','Bmpr1a','Wnt3','Bmp4','Nodal','Dll1']
ann.table.loc[[g for g in check if g in ann.table.index], ['is_tf','is_receptor','is_ligand','top_pathway']]

In [ ]:
print('sources recorded for reproducibility:')
for k, v in ann.sources.items():
    if isinstance(v, str):
        print(f'  {k}: {v[:160]}')
    else:
        print(f'  {k}: {type(v).__name__}' + (f' {getattr(v, "shape", "")}' if hasattr(v, 'shape') else ''))

## Embeddings coloured by pathway

The brief asks for embedding clusters coloured by pathway information. Each panel also carries a silhouette score and a kNN label-agreement against a label-permutation null, so "this embedding separates pathways" is a claim with a number behind it.

In [ ]:
from panelprep.embed import run_embedders
from panelprep.viz import embedding_scatter, embedding_panel, embedding_quality

embs = run_embedders({}, embedder_names=['esm2_sequence','string_ppi','kegg_causal'],
                     genes=GENES, dim=16, seed=0)
os.makedirs('outputs/annotations/figures', exist_ok=True)
pd.DataFrame([{'embedding': k, 'category': v.category, **embedding_quality(v, ann)}
              for k, v in embs.items()]).set_index('embedding')

In [ ]:
embedding_panel(embs, ann, out_path='outputs/annotations/figures/pathway_panel.png')

In [ ]:
# single large map with the role overlay
for name, emb in embs.items():
    if emb.matrix.notna().any(axis=1).sum() >= 10:
        embedding_scatter(emb, ann, color_by='top_pathway', method='pca', role_overlay=True,
                          out_path=f'outputs/annotations/figures/{name}_pathways.png')
        break

## Adding an annotation source

`panelprep/annotate/__init__.py` dispatches on the `sources` list, exactly like the score and
embedder registries. To add Reactome or a ligand-receptor database, write a function that returns
the same columns and add its name to `Config.annotation_sources`; nothing downstream changes,
because `viz.embedding_scatter` colours by whatever column you name.